# SensAI Logging & Monitoring Dashboard

Reads the same local SQLite event store as the web dashboard. Start `wu-api` and run chat interactions first. Set `MONITOR_DB_PATH` if the API uses a non-default database path.

Install optional dependencies with `pip install -e '.[monitoring]'`. Prompt and tool content may contain sensitive data; this notebook displays raw events.

In [2]:
import os
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

database_path = Path(os.getenv("MONITOR_DB_PATH", "workspace/monitoring.sqlite3"))
if not database_path.exists():
    raise FileNotFoundError(f"{database_path} does not exist. Start the API and send a chat message first.")

with sqlite3.connect(database_path) as connection:
    events = pd.read_sql_query("SELECT * FROM events ORDER BY occurred_at DESC", connection)

if events.empty:
    print("No telemetry recorded yet. Send a chat message through SensAI, then rerun this cell.")
else:
    events["occurred_at"] = pd.to_datetime(events["occurred_at"], utc=True)
    interactions = events[events["event_type"] == "interaction"].copy()
    interactions["day"] = interactions["occurred_at"].dt.floor("D")
    today = pd.Timestamp.now(tz="UTC").floor("D")
    interactions = interactions[interactions["day"] >= today - pd.Timedelta(days=13)]
    recent = interactions[interactions["day"] >= today - pd.Timedelta(days=6)]
    baseline = interactions[(interactions["day"] >= today - pd.Timedelta(days=13)) & (interactions["day"] < today - pd.Timedelta(days=6))]
    baseline_tokens = baseline["completion_tokens"].dropna().mean()
    recent_tokens = recent["completion_tokens"].dropna().mean()
    drift = ((recent_tokens - baseline_tokens) / baseline_tokens * 100) if pd.notna(baseline_tokens) and baseline_tokens else None
    llm_events = events[(events["event_type"] == "llm_call") & (events["occurred_at"] >= today - pd.Timedelta(days=13))]

    metrics = pd.Series({
        "Interactions": len(interactions),
        "Average turn latency (ms)": round(interactions["latency_ms"].dropna().mean(), 2),
        "Error rate (%)": round((1 - interactions["success"].mean()) * 100, 2),
        "Prompt tokens": int(llm_events["prompt_tokens"].fillna(0).sum()),
        "Completion tokens": int(llm_events["completion_tokens"].fillna(0).sum()),
        "Response-size drift (%)": round(drift, 2) if drift is not None else "Not enough data",
    })
    display(metrics.to_frame("Value"))
    print("Drift compares average completion tokens in the recent 7 days with the prior 7 days; it is not semantic model-quality drift.")

    days = pd.date_range(end=today, periods=14, freq="D")
    daily = interactions.groupby("day").agg(
        call_volume=("id", "count"),
        average_latency_ms=("latency_ms", "mean"),
        error_rate=("success", lambda values: 1 - values.mean()),
        average_response_tokens=("completion_tokens", "mean"),
    ).reindex(days)
    figure, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
    for axis, (title, column, color) in zip(axes.flat, [
        ("Call volume", "call_volume", "#d99b18"),
        ("Average latency (ms)", "average_latency_ms", "#8064a2"),
        ("Error rate", "error_rate", "#cc5967"),
        ("Avg completion tokens (response-size drift)", "average_response_tokens", "#2d987a"),
    ]):
        axis.plot(days, daily[column], marker="o", color=color, linewidth=2)
        axis.set_title(title)
        axis.grid(alpha=0.2)
        axis.tick_params(axis="x", rotation=35)
    axes[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
    plt.show()

    trace_columns = ["occurred_at", "event_type", "conversation_id", "model", "profile", "tool_name", "prompt", "response", "tool_arguments", "tool_result", "latency_ms", "prompt_tokens", "completion_tokens", "success", "error"]
    display(events[trace_columns].head(100))

FileNotFoundError: workspace/monitoring.sqlite3 does not exist. Start the API and send a chat message first.